# Kaggle output restore

In [2]:
!kaggle kernels output vishalmondal567/ucf-solution -p /kaggle/working/

Output file downloaded to /kaggle/working/.virtual_documents/__notebook_source__.ipynb
Output file downloaded to /kaggle/working/best_model.pth
Output file downloaded to /kaggle/working/dataset_checkpoint_test.pkl
Output file downloaded to /kaggle/working/dataset_checkpoint_train.pkl
Output file downloaded to /kaggle/working/training_state.pth
Output file downloaded to /kaggle/working/ucf-solution.log
Kernel log downloaded to /kaggle/working/ucf-solution.log 


In [3]:
import os
import cv2
import re
import numpy as np
import pickle
import multiprocessing as mp
import torch
import torch.nn as nn

from torchvision import models
import glob
from functools import partial
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader, random_split

# Config

In [4]:
# ============= CONFIGURATION =============
SEQUENCE_LENGTH = 16  # Frames per video clip
IMG_SIZE = (264, 264)
BATCH_SIZE = 84

# Class

In [5]:
class VideoFrameDataset(Dataset):
    def __init__(self, root_dir, sequence_length=16, transform=None, checkpoint_suffix=''):
        self.sequence_length = sequence_length
        self.classes = sorted(os.listdir(root_dir))
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        self.transform = transform
        self.videos = []
        
        print(f"Found classes: {self.classes}")
        
        checkpoint_file = f'/kaggle/working/dataset_checkpoint{checkpoint_suffix}.pkl'
        
        if os.path.exists(checkpoint_file):
            with open(checkpoint_file, 'rb') as f:
                checkpoint = pickle.load(f)
                self.videos = checkpoint['videos']
                self.processed_videos = checkpoint.get('processed_videos', [])
            print(f"Loaded {len(self.videos)} sequences from checkpoint")
        else:
            self.processed_videos = []
        
        # Collect tasks
        tasks = []
        for class_name in self.classes:
            class_path = os.path.join(root_dir, class_name)
            video_folders = set()
            for png in glob.glob(os.path.join(class_path, '*.png')):
                prefix = '_'.join(os.path.basename(png).split('_')[:2])
                video_folders.add(prefix)
            
            for video_folder in video_folders:
                if video_folder in self.processed_videos:
                    print(f"Skipping {video_folder} - already processed")
                    continue
                tasks.append({
                    'class_name': class_name,
                    'video_folder': video_folder,
                    'class_path': class_path,
                    'label': self.class_to_idx[class_name]
                })
        
        print(f"Processing {len(tasks)} videos using multiprocessing...")
        
        pool = mp.Pool(processes=mp.cpu_count())
        process_func = partial(self._process_video, sequence_length=sequence_length)
        
        for result in tqdm(pool.imap_unordered(process_func, tasks), total=len(tasks)):
            if result and result['sequences']:
                self.videos.extend(result['sequences'])
                self.processed_videos.append(result['video_folder'])
                
                if len(self.processed_videos) % 10 == 0:
                    with open(checkpoint_file, 'wb') as f:
                        pickle.dump({
                            'videos': self.videos,
                            'processed_videos': self.processed_videos
                        }, f)
                    print(f"✅ Checkpoint saved: {len(self.videos)} sequences")
        
        pool.close()
        pool.join()
        
        with open(checkpoint_file, 'wb') as f:
            pickle.dump({
                'videos': self.videos,
                'processed_videos': self.processed_videos
            }, f)
        print(f"Total sequences: {len(self.videos)}")
    
    def _process_video(self, task, sequence_length):
        frames = sorted(glob.glob(os.path.join(task['class_path'], f"{task['video_folder']}_*.png")))
        
        if len(frames) < sequence_length:
            return None
        
        sequences = []
        for i in range(0, len(frames) - sequence_length + 1, sequence_length // 2):
            sequences.append({
                'frames': frames[i:i+sequence_length],
                'label': task['label']
            })
        
        return {
            'video_folder': task['video_folder'],
            'sequences': sequences
        }
    
    def __len__(self):
        return len(self.videos)
    
    def __getitem__(self, idx):
        item = self.videos[idx]
        frames = []
        
        for frame_path in item['frames']:
            frame = cv2.imread(frame_path)
            frame = cv2.resize(frame, (224, 224))
            frame = frame[:, :, ::-1]  # BGR to RGB
            frame = frame / 255.0
            frame = torch.from_numpy(frame).permute(2, 0, 1).float()  # (C, H, W)
            frames.append(frame)
        
        # Stack to (T, C, H, W)
        frames = torch.stack(frames)  # (T, C, H, W)
        frames = frames.permute(1, 0, 2, 3)  # (C, T, H, W) - FIX THIS LINE
        
        label = item['label']
        return frames, label

# Model

In [6]:
# ============= CREATE MODEL WITH TRANSFER LEARNING =============
def create_video_model(num_classes, sequence_length=16):
    # Load pretrained CNN (transfer learning)
    base_cnn = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
    
    # Remove the classifier head
    base_cnn.classifier = nn.Identity()
    
    # UNFREEZE last 50% layers (CHANGE 1)
    total_layers = len(list(base_cnn.parameters()))
    freeze_until = total_layers // 2
    for i, param in enumerate(base_cnn.parameters()):
        if i < freeze_until:
            param.requires_grad = False
        else:
            param.requires_grad = True
    
    # Create full model
    class VideoAnomalyModel(nn.Module):
        def __init__(self, cnn, num_classes, seq_len):
            super().__init__()
            self.cnn = cnn
            self.seq_len = seq_len
            
            # LSTM for temporal patterns
            self.lstm = nn.LSTM(1280, 256, batch_first=True, dropout=0.3, num_layers=2, bidirectional=True)
            
            # Classifier head
            self.classifier = nn.Sequential(
                nn.Linear(512, 256),  # 512 from bidirectional LSTM
                nn.ReLU(),
                nn.Dropout(0.3),  # Reduced dropout
                nn.Linear(256, 128),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(128, num_classes)
            )
        
        def forward(self, x):
            # x shape: (batch, seq_len, C, H, W)
            batch_size, c, seq_len, h, w = x.shape  # Note: c=3, seq_len=16
            
            # Reshape to (batch * seq_len, C, H, W) for CNN
            x = x.permute(0, 2, 1, 3, 4)  # (batch, T, C, H, W)
            x = x.contiguous().view(batch_size * seq_len, c, h, w)
            
            # Extract features from each frame
            features = self.cnn(x)  # (batch * seq_len, 1280)
            
            # Reshape back to (batch, seq_len, 1280)
            features = features.view(batch_size, seq_len, -1)
            
            # LSTM for temporal patterns
            lstm_out, _ = self.lstm(features)  # (batch, seq_len, 128)
            
            # Take last timestep
            last_out = lstm_out[:, -1, :]  # (batch, 128)
            
            # Classify
            output = self.classifier(last_out)
            return output
    
    model = VideoAnomalyModel(base_cnn, num_classes, sequence_length)
    return model

# Load Data

In [8]:
# ============= LOAD DATA =============
print("Loading training data...")
full_dataset = VideoFrameDataset('/kaggle/input/datasets/odins0n/ucf-crime-dataset/Train', SEQUENCE_LENGTH, checkpoint_suffix='_train')

# Get unique video folders first
all_videos = []
for item in full_dataset.videos:
    video_id = '_'.join(item['frames'][0].split('_')[:2])  # e.g., "Abuse001"
    all_videos.append(video_id)

unique_videos = list(set(all_videos))

# Split videos, not sequences
from sklearn.model_selection import train_test_split
train_videos, val_videos = train_test_split(unique_videos, test_size=0.2, random_state=42)

# Filter dataset indices
train_indices = []
val_indices = []

for idx, item in enumerate(full_dataset.videos):
    video_id = '_'.join(item['frames'][0].split('_')[:2])
    if video_id in train_videos:
        train_indices.append(idx)
    else:
        val_indices.append(idx)

# Create subset datasets
from torch.utils.data import Subset
train_dataset = Subset(full_dataset, train_indices)
val_dataset = Subset(full_dataset, val_indices)

print(f"Train videos: {len(train_videos)}, Val videos: {len(val_videos)}")
print(f"Train sequences: {len(train_indices)}, Val sequences: {len(val_indices)}")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

Loading training data...
Found classes: ['Abuse', 'Arrest', 'Arson', 'Assault', 'Burglary', 'Explosion', 'Fighting', 'NormalVideos', 'RoadAccidents', 'Robbery', 'Shooting', 'Shoplifting', 'Stealing', 'Vandalism']
Loaded 155986 sequences from checkpoint
Skipping Abuse012_x264 - already processed
Skipping Abuse038_x264 - already processed
Skipping Abuse049_x264 - already processed
Skipping Abuse024_x264 - already processed
Skipping Abuse031_x264 - already processed
Skipping Abuse048_x264 - already processed
Skipping Abuse009_x264 - already processed
Skipping Abuse029_x264 - already processed
Skipping Abuse034_x264 - already processed
Skipping Abuse046_x264 - already processed
Skipping Abuse036_x264 - already processed
Skipping Abuse015_x264 - already processed
Skipping Abuse020_x264 - already processed
Skipping Abuse010_x264 - already processed
Skipping Abuse037_x264 - already processed
Skipping Abuse033_x264 - already processed
Skipping Abuse001_x264 - already processed
Skipping Abuse05

100%|██████████| 3/3 [00:02<00:00,  1.18it/s]


Total sequences: 155986
Train videos: 1285, Val videos: 322
Train sequences: 129418, Val sequences: 26568


In [9]:
# ============= TRAIN MODEL =============
num_classes = len(full_dataset.classes)
model = create_video_model(num_classes, SEQUENCE_LENGTH)

# Move to GPU with parallelism
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)
model = model.to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()

# Train ALL unfrozen layers (CNN + LSTM + classifier)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)  # Remove the filter

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

start_epoch = 0
best_val_acc = 0
best_val_loss = float('inf')
patience_counter = 0

EPOCHS = 30

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 119MB/s]


Using 2 GPUs


In [10]:
checkpoint_path = '/kaggle/working/best_model.pth'
if os.path.exists(checkpoint_path):
    print(f"Loading checkpoint from {checkpoint_path}")
    if isinstance(model, nn.DataParallel):
        model.module.load_state_dict(torch.load(checkpoint_path))
    else:
        model.load_state_dict(torch.load(checkpoint_path))
    
    # Optional: Load full training state (epoch, optimizer, scheduler)
    full_checkpoint = '/kaggle/working/training_state.pth'
    if os.path.exists(full_checkpoint):
        state = torch.load(full_checkpoint)
        start_epoch = state['epoch'] + 1
        best_val_acc = state['best_val_acc']
        best_val_loss = state['best_val_loss']
        patience_counter = state['patience_counter']
        optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler'])
        print(f"Resuming from epoch {start_epoch}")
        print(best_val_acc)
        print(best_val_loss)
        print(patience_counter)

# Training

In [11]:
# Training loop
for epoch in range(start_epoch, EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_idx, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        if batch_idx % 100 == 0:
            print(f'Epoch: {epoch}, Batch: {batch_idx}, Loss: {loss.item():.4f}')


    torch.save({
        'epoch': epoch,
        'best_val_acc': best_val_acc,
        'best_val_loss': best_val_loss,
        'patience_counter': patience_counter,
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
    }, '/kaggle/working/training_state.pth')
    
    
    # Validation
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
    
    train_acc = 100. * correct / total
    val_acc = 100. * val_correct / val_total

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= 5:
            print(f"Early stopping at epoch {epoch}")
            break

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        if isinstance(model, nn.DataParallel):
            torch.save(model.module.state_dict(), '/kaggle/working/best_model.pth')
        else:
            torch.save(model.state_dict(), '/kaggle/working/best_model.pth')
        print(f"✅ Best model saved with val_acc: {val_acc:.2f}%")
    
    print(f'Epoch: {epoch}')
    print(f'Train Loss: {total_loss/len(train_loader):.4f}, Train Acc: {train_acc:.2f}%')
    print(f'Val Loss: {val_loss/len(val_loader):.4f}, Val Acc: {val_acc:.2f}%')
    
    scheduler.step(val_loss)

print("✅ Training complete!")

Epoch: 0, Batch: 0, Loss: 2.6271
Epoch: 0, Batch: 100, Loss: 1.0118
Epoch: 0, Batch: 200, Loss: 0.5040
Epoch: 0, Batch: 300, Loss: 0.6441
Epoch: 0, Batch: 400, Loss: 0.4789
Epoch: 0, Batch: 500, Loss: 0.3382
Epoch: 0, Batch: 600, Loss: 0.4701
Epoch: 0, Batch: 700, Loss: 0.3111
Epoch: 0, Batch: 800, Loss: 0.3880
Epoch: 0, Batch: 900, Loss: 0.2279
Epoch: 0, Batch: 1000, Loss: 0.2781
Epoch: 0, Batch: 1100, Loss: 0.3528
Epoch: 0, Batch: 1200, Loss: 0.3052
Epoch: 0, Batch: 1300, Loss: 0.1393
Epoch: 0, Batch: 1400, Loss: 0.0981
Epoch: 0, Batch: 1500, Loss: 0.1295
✅ Best model saved with val_acc: 63.78%
Epoch: 0
Train Loss: 0.4028, Train Acc: 88.13%
Val Loss: 1.5816, Val Acc: 63.78%
Epoch: 1, Batch: 0, Loss: 0.1830


KeyboardInterrupt: 

# Testing

In [12]:
# For test dataset (no labels, just prediction)
test_dataset = VideoFrameDataset('/kaggle/input/datasets/odins0n/ucf-crime-dataset/Test', SEQUENCE_LENGTH, checkpoint_suffix='_test')

num_classes = len(test_dataset.classes)
model = create_video_model(num_classes, SEQUENCE_LENGTH)

# Create DataLoader
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=False)

# Load best model - handle DataParallel
checkpoint = torch.load('/kaggle/working/best_model.pth')

# Remove 'module.' prefix if checkpoint was saved with DataParallel
if any(k.startswith('module.') for k in checkpoint.keys()):
    from collections import OrderedDict
    new_state_dict = OrderedDict()
    for k, v in checkpoint.items():
        name = k[7:]  # remove 'module.'
        new_state_dict[name] = v
    checkpoint = new_state_dict

model.load_state_dict(checkpoint)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)
    
model = model.to(device)
model.eval()

# Test
test_correct = 0
test_total = 0
test_top5_correct = 0
all_preds = []
all_labels = []

def compute_top5_accuracy(outputs, labels):
    _, top5_pred = outputs.topk(5, 1, True, True)
    top5_correct = top5_pred.eq(labels.view(-1, 1).expand_as(top5_pred))
    return top5_correct.any(dim=1).float().sum().item()

# Validation
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        
        # Top-1 accuracy
        _, predicted = outputs.max(1)
        test_correct += predicted.eq(labels).sum().item()
        
        # Top-5 accuracy
        test_top5_correct += compute_top5_accuracy(outputs, labels)
        
        test_total += labels.size(0)

test_acc_top1 = 100. * test_correct / test_total
test_acc_top5 = 100. * test_top5_correct / test_total
print(f'Test Top-1: {test_acc_top1:.2f}%, Test Top-5: {test_acc_top5:.2f}%')

Found classes: ['Abuse', 'Arrest', 'Arson', 'Assault', 'Burglary', 'Explosion', 'Fighting', 'NormalVideos', 'RoadAccidents', 'Robbery', 'Shooting', 'Shoplifting', 'Stealing', 'Vandalism']
Loaded 13714 sequences from checkpoint
Skipping Abuse028_x264 - already processed
Skipping Abuse030_x264 - already processed
Skipping Arrest001_x264 - already processed
Skipping Arrest039_x264 - already processed
Skipping Arrest030_x264 - already processed
Skipping Arrest007_x264 - already processed
Skipping Arrest024_x264 - already processed
Skipping Arson041_x264 - already processed
Skipping Arson011_x264 - already processed
Skipping Arson035_x264 - already processed
Skipping Arson018_x264 - already processed
Skipping Arson007_x264 - already processed
Skipping Arson009_x264 - already processed
Skipping Arson016_x264 - already processed
Skipping Arson010_x264 - already processed
Skipping Arson022_x264 - already processed
Skipping Assault010_x264 - already processed
Skipping Assault011_x264 - already 

0it [00:00, ?it/s]


Total sequences: 13714
Using 2 GPUs
Test Top-1: 50.88%, Test Top-5: 73.95%
